# 08. Optical-Image Segmentation with a U-Net-Style Baseline

U-Net remains an excellent first baseline for many biomedical segmentation problems because it combines local features with multiscale context.

In [ ]:
import torch
from torch import nn

class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.Conv2d(16,16,3,padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1), nn.ReLU())
        self.up = nn.ConvTranspose2d(32,16,2,stride=2)
        self.dec = nn.Sequential(nn.Conv2d(32,16,3,padding=1), nn.ReLU(), nn.Conv2d(16,1,1))
    def forward(self, x):
        s = self.enc1(x)
        z = self.enc2(self.pool(s))
        z = self.up(z)
        return self.dec(torch.cat([z, s], dim=1))

model = TinyUNet()
print(model(torch.rand(2,1,64,64)).shape)

## BCE + Dice starting point

Use logits during training; apply sigmoid only when probabilities are needed for inspection/thresholding.

In [ ]:
def dice_loss_from_logits(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1,2,3))
    denom = p.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (denom + eps)
    return 1 - dice.mean()

bce = nn.BCEWithLogitsLoss()
logits = torch.randn(2,1,64,64)
target = (torch.rand(2,1,64,64) > .7).float()
loss = bce(logits, target) + dice_loss_from_logits(logits, target)
print(float(loss))

## Validation rule

Do not tune a segmentation threshold on the test set. Inspect object-level failures and performance by specimen, not only a pooled pixel metric.